# Data Cleaning – IRA Troll Factory Tweets (PySpark)

**Course:** Big Data and Smart Data Analytics

**Assignment:** Analyze 3 million tweets produced by the IRA troll factory in the 2012-2018 period -  Data Pre-processing and Cleaning 

**Students:** Bastien Goiffon , Priscille Montoussé , Sara Silva

**Date:** 10/12/2025


## 1. Introduction to the notebook

We load and examine the raw dataset (about 3 million tweets) using **PySpark**.  More specifically, we apply this code to a **merged document version** of the 13 csv files uploaded on github available in https://github.com/fivethirtyeight/russian-troll-tweets/ (tweets from accounts associated with the Internet Research Agency). This notebook documents **all of the records of the data cleaning process** and includes:

1. Loading and inspecting the merged dataset.
2. Correction of mislabelled columns.
3. Conversion of variables to proper data types.
4. Examination and dealing with missing values
5. Simplifying and engineering features
6. Cleaning categorical and textual variables
7. Removal of duplicates and check data consistency
8. Creation of a dataset cleaned version - parquet and delta table format


Throughout the notebook, we show our data-cleaning pipeline and we include clear explanations of the rationale of each step in markdown cells.


## 2. Setup and Data Loading

In this section we configure the **Spark session** and import the **PySpark functions** used throughout the notebook.
We also load our merged dataset `merged_IRA_tweets_github_full.csv` and, to confirm that the CSV has been appropriately parsed, we examine the schema and display an example row.

In [1]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import to_timestamp, coalesce
from pyspark.sql.functions import sum as _sum, when
from pyspark.sql.functions import lower, trim, regexp_replace

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 3, Finished, Available, Finished)

In [2]:
file_path = "Files/merged_IRA_tweets_github_full.csv"

df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True,
    sep=",",
    quote="\"",
    escape="\"",
    multiLine=True
)

print("CSV loaded with proper quoting.")
df.show(1, vertical=True, truncate=False)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 4, Finished, Available, Finished)

CSV loaded with proper quoting.
-RECORD 0--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 external_author_id | 906000000000000000                                                                                                                                           
 author             | 10_GOP                                                                                                                                                       
 content            | "We have a sitting Democrat US Senator on trial for corruption and you've barely heard a peep from the mainstream media." ~ @nedryun https://t.co/gh6g0D1oiC 
 region             | Unknown                                                                                                                                                      
 language           | English                                       

## 3. Detecting Mislabelled Columns

Before proceeding, we look at the first rows of the dataset and notice an important discrepancy in the column headers: The column `retweet` , which should contain a 0/1 indicator, instead contains text labels such as _RightTroll_, _LeftTroll_, _NewsFeed_. According to Linvill & Warren, these numbers fall into the general account category. The column `account_category`, which should contain the text labels, instead contains the numeric values - the `retweet` flag (retweet = 1, original tweet = 0).

This implies that the **two columns' header names and the data beneath them are switched**.
In order to fix this problem, we: 
1. Rename the variables with wrong labels into temporary columns.
2. Cast them to the appropriate data types.
3. Change the column names to reflect the correct meaning.

After correcting the mislabelled columns, we perform a quick structural validation of the dataset.  


In [3]:
df = df.withColumn("retweet_tmp", col("account_category").cast(IntegerType()))
df = df.withColumn("account_category_tmp", col("retweet").cast("string"))

df = (
    df
    .drop("retweet", "account_category")
    .withColumnRenamed("retweet_tmp", "retweet")
    .withColumnRenamed("account_category_tmp", "account_category")
)

print("After swapping columns:")
df.show(1, vertical=True, truncate=False)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 5, Finished, Available, Finished)

After swapping columns:
-RECORD 0--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 external_author_id | 906000000000000000                                                                                                                                           
 author             | 10_GOP                                                                                                                                                       
 content            | "We have a sitting Democrat US Senator on trial for corruption and you've barely heard a peep from the mainstream media." ~ @nedryun https://t.co/gh6g0D1oiC 
 region             | Unknown                                                                                                                                                      
 language           | English                                               

These next computations ensure that the dataset has been fully loaded, no rows were lost during the column-swapping operation, column names are defined correctly and the number of the variables matches the initial schema.

In [4]:
row_count = df.count()
print(f"Total Number of Rows (Tweets): {row_count}")

col_count = len(df.columns)
print(f"Total Number of Columns (Variables): {col_count}")

print("List of all Column Names:")
for col_name in df.columns:
    print(f"- {col_name}")

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 6, Finished, Available, Finished)

Total Number of Rows (Tweets): 2946207
Total Number of Columns (Variables): 21
List of all Column Names:
- external_author_id
- author
- content
- region
- language
- publish_date
- harvested_date
- following
- followers
- updates
- post_type
- account_type
- new_june_2018
- alt_external_id
- tweet_id
- article_url
- tco1_step1
- tco2_step1
- tco3_step1
- retweet
- account_category


In [5]:
print("DataFrame Schema (Variable Names and Types):")
df.printSchema()

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 7, Finished, Available, Finished)

DataFrame Schema (Variable Names and Types):
root
 |-- external_author_id: string (nullable = true)
 |-- author: string (nullable = true)
 |-- content: string (nullable = true)
 |-- region: string (nullable = true)
 |-- language: string (nullable = true)
 |-- publish_date: string (nullable = true)
 |-- harvested_date: string (nullable = true)
 |-- following: integer (nullable = true)
 |-- followers: integer (nullable = true)
 |-- updates: integer (nullable = true)
 |-- post_type: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- new_june_2018: integer (nullable = true)
 |-- alt_external_id: string (nullable = true)
 |-- tweet_id: long (nullable = true)
 |-- article_url: string (nullable = true)
 |-- tco1_step1: string (nullable = true)
 |-- tco2_step1: string (nullable = true)
 |-- tco3_step1: string (nullable = true)
 |-- retweet: integer (nullable = true)
 |-- account_category: string (nullable = true)



In [6]:
df.show(10, truncate=False)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 8, Finished, Available, Finished)

+------------------+------+------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+--------+---------------+---------------+---------+---------+-------+---------+------------+-------------+------------------+------------------+-----------------------------------------------------------------+---------------------------------------------------------------------+----------+----------+-------+----------------+
|external_author_id|author|content                                                                                                                                                     |region |language|publish_date   |harvested_date |following|followers|updates|post_type|account_type|new_june_2018|alt_external_id   |tweet_id          |article_url                                                      |tco1_step1                                                           |tco2_ste

## 4. Converting Variables to Proper Data Types

Before performing any cleaning, transformation or analysis, we convert each variable to its correct data type. This improves memory utilization and execution performance, the precision of filtering and grouping operations, and the reliability of time-series analysis.

- We need numerous patterns to parse the dates in the columns `publish_date` and `harvested_date` because they are in two somewhat different formats. Both are converted to the Spark `timestamp` type.
- `following`, `followers`, `updates`, `new_june_2018` (0/1 indicator) and `retweet` (0/1 retweet flag) represent counts or binary indicators and must be stored as integers.
- Spark’s Boolean type can be inconvenient for filtering and grouping operations, so we keep `retweet` and `new_june_2018` as integers instead of converting them to Boolean.
- These next variables are identifiers, categories or free-text fields, so we keep them as strings:
1. IDs: `external_author_id`, `alt_external_id`, `tweet_id`
2. Author metadata: `author`, `region`, `language`, `account_type`, `account_category`
3. Tweet text: `content`
4. URL fields: `article_url`, `tco1_step1`, `tco2_step1`, `tco3_step1`
5. Other string fields: `post_type`


In [7]:
df = df.withColumn(
    "publish_date",
    coalesce(
        to_timestamp(col("publish_date").cast("string"), "M/d/yyyy H:mm"),
        to_timestamp(col("publish_date").cast("string"), "MM/dd/yyyy HH:mm")
    )
)

df = df.withColumn(
    "harvested_date",
    coalesce(
        to_timestamp(col("harvested_date").cast("string"), "M/d/yyyy H:mm"),
        to_timestamp(col("harvested_date").cast("string"), "MM/dd/yyyy HH:mm")
    )
)

int_cols = ["following", "followers", "updates", "new_june_2018", "retweet"]

for c in int_cols:
    df = df.withColumn(c, col(c).cast(IntegerType()))

print("Schema after final conversions:")
df.printSchema()
df.show(1, vertical=True, truncate=False)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 9, Finished, Available, Finished)

Schema after final conversions:
root
 |-- external_author_id: string (nullable = true)
 |-- author: string (nullable = true)
 |-- content: string (nullable = true)
 |-- region: string (nullable = true)
 |-- language: string (nullable = true)
 |-- publish_date: timestamp (nullable = true)
 |-- harvested_date: timestamp (nullable = true)
 |-- following: integer (nullable = true)
 |-- followers: integer (nullable = true)
 |-- updates: integer (nullable = true)
 |-- post_type: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- new_june_2018: integer (nullable = true)
 |-- alt_external_id: string (nullable = true)
 |-- tweet_id: long (nullable = true)
 |-- article_url: string (nullable = true)
 |-- tco1_step1: string (nullable = true)
 |-- tco2_step1: string (nullable = true)
 |-- tco3_step1: string (nullable = true)
 |-- retweet: integer (nullable = true)
 |-- account_category: string (nullable = true)

-RECORD 0-------------------------------------------------------

## 5. Quantifying Missing Values

Before deciding how to clean the dataset, it is important to understand how complete each variable is.  

In [8]:
missing_exprs = []
for c in df.columns:
    missing_exprs.append(
        _sum(
            when(col(c).isNull() | (col(c) == ""), 1).otherwise(0)
        ).alias(c)
    )

missing_df = df.agg(*missing_exprs)

missing_long = missing_df.selectExpr(
    "stack({0}, {1}) as (column, missing_count)".format(
        len(df.columns),
        ", ".join([f"'{c}', `{c}`" for c in df.columns])
    )
)

total_rows = df.count()

missing_long = (
    missing_long
    .withColumn("pct_missing", (col("missing_count") / total_rows) * 100)
    .orderBy(col("missing_count").desc())
)

missing_long.show(truncate=False)


StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 10, Finished, Available, Finished)

+------------------+-------------+---------------------+
|column            |missing_count|pct_missing          |
+------------------+-------------+---------------------+
|tco3_step1        |2931207      |99.49087080439358    |
|tco2_step1        |2235090      |75.86330492053003    |
|post_type         |1648625      |55.95754134044213    |
|tco1_step1        |845971       |28.713902315757174   |
|region            |8818         |0.29930008312382667  |
|content           |1            |3.3941946373761244E-5|
|external_author_id|0            |0.0                  |
|author            |0            |0.0                  |
|language          |0            |0.0                  |
|publish_date      |0            |0.0                  |
|harvested_date    |0            |0.0                  |
|following         |0            |0.0                  |
|followers         |0            |0.0                  |
|updates           |0            |0.0                  |
|account_type      |0          

There are **no missing data** in fields like `author`, `language`, `publish_date`, `followers`, `tweet_id`, and other essential attributes. This is to be expected since these fields are regularly filled in and originate directly from **Twitter's metadata**.

The optional URL-related fields (`tco1_step1`, `tco2_step1`, `tco3_step1`) and in `post_type` present a different scenario. In certain instances, the missingness in these columns is **over 90%**. Their sparsity is not an issue because they come from auxiliary URL-expansion operations and are not necessary for our study.

Conversely, certain variables show very little missingness. For instance, only one tweet lacks the `content` field, and the proportion of null values in the `region` field is extremely low. If needed, these can be adjusted or imputed without changing the dataset.

While some peripheral metadata fields are either inconsistently collected or only present for certain tweets, the missing-value profile often verifies that the **essential tweet information is intact**.


`Retweet` vs. `post_type`

One variable in particular stood out during the missing-values analysis: `post_type`.  
The question of whether **this field is significant or trustworthy enough to maintain** is raised by the fact that more than half of its entries are missing.

In order to make its utility clear, it must be compared to the related variable `retweet`, which is complete and well-behaved. Despite their similar names, these two fields represent distinct ideas and were **produced by different methods in the original dataset**. The seemingly identical fields `retweet` and `post_type` do not measure the same thing, and their discrepancy is not a mistake. It has to do with how Twitter initially produced these variables during the extraction of the dataset.

Twitter's own metadata is the source of the `retweet` column. It uses Twitter's built-in retweet mechanism and is a straightforward and trustworthy 0/1 indicator that indicates whether the tweet object itself is a retweet. This field is comprehensive, reliable, and consistent.

However, a different extraction pipeline that uses redirect parsing and URL expansion generated the `post_type` information. Labels like "RETWEET," "QUOTE_TWEET," or "TWEET" are usually returned. The success of the URL-expansion stage for each tweet, however, is crucial. The field is left blank whenever this procedure was unsuccessful or omitted. A blank value just indicates that **"Twitter did not classify this tweet," not that the tweet is unique**.

Only three states are actually present in the column: "RETWEET," "QUOTE_TWEET," and blank/null. The field is unreliable for analytical work because over half of its entries are missing, and categorization errors are indistinguishable from genuine "original tweets."

Adding such an inconsistent variable will add needless noise because the main goals of our research are to identify tweet behavior, categorize accounts, and analyze interaction patterns. Because of these factors, `post_type` **is eliminated from the dataset since it serves no benefit for us.**


In [9]:
cols_to_drop = ["post_type"]

df = df.drop(*cols_to_drop)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 11, Finished, Available, Finished)

The dataset includes up to three optional URL fields (`tco1_step1`, `tco2_step1`, and `tco3_step1`). These represent the initial enlarged version of each shortened `t.co` link and come from Twitter's redirect-resolution procedure.

However, these fields are rather sparse. The majority of tweets lack one or more of these columns, and the three columns merely indicate the presence of a link—not its content. We combine their data into a single, easier-to-understand feature rather than maintaining them as distinct, largely empty variables.

We generate **`n_links`**, which merely counts the number of each tweet's three URL fields. This results in a condensed, useful variable that expresses how many outbound links a tweet has. After, we remove the three original columns, keeping only the new numerical summary feature.

In [10]:
#Create a new column n_links
df = df.withColumn(
    "n_links",
    (col("tco1_step1").isNotNull().cast("int") +
     col("tco2_step1").isNotNull().cast("int") +
     col("tco3_step1").isNotNull().cast("int"))
)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 12, Finished, Available, Finished)

In [11]:
df = df.drop("tco1_step1", "tco2_step1", "tco3_step1")

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 13, Finished, Available, Finished)

In [12]:
print("Dataset Schema (Variables and Types):")
df.printSchema()

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 14, Finished, Available, Finished)

Dataset Schema (Variables and Types):
root
 |-- external_author_id: string (nullable = true)
 |-- author: string (nullable = true)
 |-- content: string (nullable = true)
 |-- region: string (nullable = true)
 |-- language: string (nullable = true)
 |-- publish_date: timestamp (nullable = true)
 |-- harvested_date: timestamp (nullable = true)
 |-- following: integer (nullable = true)
 |-- followers: integer (nullable = true)
 |-- updates: integer (nullable = true)
 |-- account_type: string (nullable = true)
 |-- new_june_2018: integer (nullable = true)
 |-- alt_external_id: string (nullable = true)
 |-- tweet_id: long (nullable = true)
 |-- article_url: string (nullable = true)
 |-- retweet: integer (nullable = true)
 |-- account_category: string (nullable = true)
 |-- n_links: integer (nullable = false)



In [13]:
#Checking missing values again
missing_exprs = [
    _sum( when(col(c).isNull() | (col(c) == ""), 1).otherwise(0) ).alias(c)
    for c in df.columns
]

missing_df = df.agg(*missing_exprs)

missing_long = (
    missing_df
    .selectExpr(
        "stack({0}, {1}) as (column, missing_count)".format(
            len(df.columns),
            ", ".join([f"'{c}', `{c}`" for c in df.columns])
        )
    )
)

total_rows = df.count()

missing_long = (
    missing_long
    .withColumn("pct_missing", (col("missing_count") / total_rows) * 100)
    .orderBy(col("missing_count").desc())
)

missing_long.show(truncate=False)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 15, Finished, Available, Finished)

+------------------+-------------+---------------------+
|column            |missing_count|pct_missing          |
+------------------+-------------+---------------------+
|region            |8818         |0.29930008312382667  |
|content           |1            |3.3941946373761244E-5|
|external_author_id|0            |0.0                  |
|author            |0            |0.0                  |
|language          |0            |0.0                  |
|publish_date      |0            |0.0                  |
|harvested_date    |0            |0.0                  |
|following         |0            |0.0                  |
|followers         |0            |0.0                  |
|updates           |0            |0.0                  |
|account_type      |0            |0.0                  |
|new_june_2018     |0            |0.0                  |
|alt_external_id   |0            |0.0                  |
|tweet_id          |0            |0.0                  |
|article_url       |0          

Nearly all meaningful missingness is now absent from the dataset. There are gaps in just two columns:

- Due to the fact that many accounts were not given a geographic category, the **`region` contains about 0.29% missing entries**.

- Out of about three million tweets, there is one missing entry in the `content`, which is practically nothing.

All other variables are now fully complete. This confirms that the remaining missing values are minimal, non-problematic, and do not affect downstream analysis.

## 6. Standardising Categorical Labels

Some categorical variables continue to exhibit internal discrepancies. This resulted from the fact that **different spellings or placeholder values** could appear in the same category during the initial data collection procedure. We standardized some of these remaining cases to prevent classifying similar labels as distinct groups in the future.

In [14]:
cols_to_fix = ["region", "language", "account_type", "account_category", "author"]

for c in cols_to_fix:
    df = df.withColumn(c, trim(lower(df[c])))

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 16, Finished, Available, Finished)

In [15]:
#Checking unique values
categorical_cols = ["region", "language", "account_type", "account_category", "author"]

for c in categorical_cols:
    print(f"\n--- Unique values in {c} ---")
    df.select(c).distinct().orderBy(c).show(100, truncate=False)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 17, Finished, Available, Finished)


--- Unique values in region ---
+-------------------------+
|region                   |
+-------------------------+
|NULL                     |
|afghanistan              |
|austria                  |
|azerbaijan               |
|belarus                  |
|canada                   |
|czech republic           |
|denmark                  |
|egypt                    |
|estonia                  |
|finland                  |
|france                   |
|germany                  |
|greece                   |
|hong kong                |
|india                    |
|iran, islamic republic of|
|iraq                     |
|israel                   |
|italy                    |
|japan                    |
|latvia                   |
|malaysia                 |
|mexico                   |
|russian federation       |
|samoa                    |
|saudi arabia             |
|serbia                   |
|spain                    |
|sweden                   |
|switzerland              |
|turkey        

In `region`, NULL and "unknown" appeared. For consistency, we replace missing values with "unknown".

We also map "language undefined" to "unknown" for a few entries in the `language`.

We also substitute "unknown" for the "?" that is used to label some data in `account_type` as it does not provide a useful category.

These modifications guarantee that the categorical variables exhibit **consistent behavior** when utilized for modeling, grouping, and visualization.

In [16]:
# REGION: NULL → "unknown"
df = df.withColumn(
    "region",
    when(col("region").isNull(), "unknown").otherwise(col("region"))
)

# LANGUAGE: "language undefined" → "unknown"
df = df.withColumn(
    "language",
    when(col("language") == "language undefined", "unknown").otherwise(col("language"))
)

# ACCOUNT_TYPE: "?" → "unknown"
df = df.withColumn(
    "account_type",
    when(col("account_type") == "?", "unknown").otherwise(col("account_type"))
)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 18, Finished, Available, Finished)

In [17]:
#Verifying final unique values
df.select("region").distinct().orderBy("region").show(100, truncate=False)
df.select("language").distinct().orderBy("language").show(100, truncate=False)
df.select("account_type").distinct().orderBy("account_type").show(100, truncate=False)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 19, Finished, Available, Finished)

+-------------------------+
|region                   |
+-------------------------+
|afghanistan              |
|austria                  |
|azerbaijan               |
|belarus                  |
|canada                   |
|czech republic           |
|denmark                  |
|egypt                    |
|estonia                  |
|finland                  |
|france                   |
|germany                  |
|greece                   |
|hong kong                |
|india                    |
|iran, islamic republic of|
|iraq                     |
|israel                   |
|italy                    |
|japan                    |
|latvia                   |
|malaysia                 |
|mexico                   |
|russian federation       |
|samoa                    |
|saudi arabia             |
|serbia                   |
|spain                    |
|sweden                   |
|switzerland              |
|turkey                   |
|ukraine                  |
|united arab emirate

## 7. Checking for Full-Row Duplicates

We check to see whether there are any duplicate observations in the combined file. Verifying that no tweet appears more than once after merging is crucial because the dataset is derived from various CSV files and spans several years of activity.

The number of distinct rows and the total number of rows are compared.
**There are no full-row duplicates** in the dataset because both counts match exactly.
This indicates that the merge was successful and that no deduplication step is needed for the dataset. However, we check for duplicates rows again later in the notebook, where we found duplicated `tweet_id`.

In [18]:
# Count total rows
total_rows = df.count()

# Count distinct rows
distinct_rows = df.dropDuplicates().count()

# Number of duplicate rows
duplicates = total_rows - distinct_rows

print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Full-row duplicates:", duplicates)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 20, Finished, Available, Finished)

Total rows: 2946207
Distinct rows: 2946207
Full-row duplicates: 0


## 8. Inspecting Outliers in Numeric Columns

We investigate the fundamental descriptive statistics of the primary activity metrics (followers, following, and updates) in order to comprehend their behavior. The distributions are **significantly skewed**,**as is typical for social media datasets**: a small percentage of accounts exhibit extraordinarily high values, while the rest stay at much lower levels. This pattern is common on social media sites like Twitter, where activity and visibility tend to focus on a limited number of accounts.

In [19]:
#Basic descriptive statistics
df.select("followers", "following", "updates").describe().show()

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 21, Finished, Available, Finished)

+-------+------------------+------------------+------------------+
|summary|         followers|         following|           updates|
+-------+------------------+------------------+------------------+
|  count|           2946207|           2946207|           2946207|
|   mean| 7055.265491868019|3448.7278531345555|10514.685623583136|
| stddev|14635.939344600809| 5625.644178869655| 17705.54568943745|
|    min|                -1|                -1|                -1|
|    max|            251276|             76210|            166113|
+-------+------------------+------------------+------------------+



Although the summary statistics clearly point to heavy tails, a simple min–max analysis is insufficient since high values in this dataset can represent real behavior rather than noise. In order to determine how rare the greatest numbers are, we look at **percentile thresholds**.

We verify that the **maximum observed values lie precisely at the top of the distribution** and do not beyond the predicted range using the 99.9th and 99.99th percentiles. To put it another way, there are no erroneous or contrived outliers over the 99.9th percentile in the dataset. Real accounts created to maximize reach, activity, and automated interactions are represented by these extremely high values.

One problem did arise, though: the **minimum of all three numerical variables was -1**, which is not a significant figure for update, follower, or following counts. In order to prevent adding inaccurate data to the analysis, these negative values are handled as placeholders and substituted with NULL.

We verify that all faulty values have been eliminated by rechecking the dataset after the -1 entries have been replaced.

In [20]:
#Percentile-based outlier detection
quantiles = df.approxQuantile(
    ["followers", "following", "updates"],
    [0.5, 0.9, 0.99, 0.999],
    0.01
)

print(quantiles)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 22, Finished, Available, Finished)

[[1207.0, 19473.0, 251276.0, 251276.0], [1482.0, 8813.0, 76210.0, 76210.0], [4319.0, 27410.0, 166113.0, 166113.0]]


In [21]:
#Count how many outliers exist
# Get percentile thresholds
p_followers = df.approxQuantile("followers", [0.999], 0.01)[0]
p_following = df.approxQuantile("following", [0.999], 0.01)[0]
p_updates   = df.approxQuantile("updates",   [0.999], 0.01)[0]

print("OUTLIER THRESHOLDS:")
print("followers >", p_followers)
print("following >", p_following)
print("updates   >", p_updates)

# Count outliers
df.filter(df.followers > p_followers).count(), \
df.filter(df.following > p_following).count(), \
df.filter(df.updates   > p_updates).count()

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 23, Finished, Available, Finished)

OUTLIER THRESHOLDS:
followers > 251276.0
following > 76210.0
updates   > 166113.0


(0, 0, 0)

In [22]:
num_cols = ["followers", "following", "updates"]

for c in num_cols:
    df = df.withColumn(c, when(col(c) == -1, None).otherwise(col(c)))

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 24, Finished, Available, Finished)

In [23]:
#Checking that the -1 values are gone
for c in ["followers", "following", "updates"]:
    print(c, " → ", df.filter(col(c) == -1).count())

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 25, Finished, Available, Finished)

followers  →  0
following  →  0
updates  →  0


In [24]:
#Rechecking missing values
missing_exprs = []

for c in df.columns:
    missing_exprs.append(
        _sum(when(col(c).isNull() | (col(c) == ""), 1).otherwise(0)).alias(c)
    )

missing_df = df.agg(*missing_exprs)

from pyspark.sql.functions import expr

missing_long = missing_df.selectExpr(
    "stack({0}, {1}) as (column, missing_count)".format(
        len(df.columns),
        ", ".join([f"'{c}', `{c}`" for c in df.columns])
    )
).orderBy(col("missing_count").desc())

total_rows = df.count()
missing_long = missing_long.withColumn(
    "pct_missing", col("missing_count") / total_rows * 100
)

missing_long.show(truncate=False)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 26, Finished, Available, Finished)

+------------------+-------------+---------------------+
|column            |missing_count|pct_missing          |
+------------------+-------------+---------------------+
|following         |3            |1.0182583912128374E-4|
|followers         |3            |1.0182583912128374E-4|
|updates           |3            |1.0182583912128374E-4|
|content           |1            |3.3941946373761244E-5|
|external_author_id|0            |0.0                  |
|author            |0            |0.0                  |
|region            |0            |0.0                  |
|language          |0            |0.0                  |
|publish_date      |0            |0.0                  |
|harvested_date    |0            |0.0                  |
|account_type      |0            |0.0                  |
|new_june_2018     |0            |0.0                  |
|alt_external_id   |0            |0.0                  |
|tweet_id          |0            |0.0                  |
|article_url       |0          

## 9. Cleaning the Main Text Content

The `content` column contains the raw text of each tweet, which has a number of formatting errors because it was taken from several sources and tools. We must **standardize** this text before utilizing it for any further investigation.

Invisible characters like line breaks (\n), carriage returns (\r), and tab spaces (\t) are eliminated during the cleaning procedure. Additionally, it eliminates extraneous spacing at the start or finish of the string and replaces sequences of numerous spaces with a single one. Lastly, a **straightforward placeholder** is used in place of any tweet that ends up empty—usually tweets that just included a retweet marker or other non-text artifacts.

This guarantees that missing `content` is handled in a predictable manner and that **all text fields adhere to a uniform structure**.

In [25]:
# 1) Remove line breaks \r, \n, and tabs \t
df = df.withColumn(
    "content",
    regexp_replace(col("content"), "[\r\n\t]", " ")
)

# 2) Replace multiple spaces with a single space
df = df.withColumn(
    "content",
    regexp_replace(col("content"), " +", " ")
)

# 3) Trim spaces at beginning or end (optional but recommended)

df = df.withColumn("content", trim(col("content")))

# 4) Replace null or empty content with a placeholder
df = df.withColumn(
    "content",
    when(col("content").isNull() | (col("content") == ""), "[empty]").otherwise(col("content"))
)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 27, Finished, Available, Finished)

## 10. Check for Duplicate `tweet_id` Values

To verify whether the ID for each tweet uniquely identifies it, we group the dataset by `tweet_id` and counted occurrences. The initial inspection immediately shows that **`tweet_id` is not unique**. Many IDs appear twice, and the top duplicated entries all have a count of exactly 2.

In [26]:
df.groupBy("tweet_id").count().filter(col("count") > 1).show(20, truncate=False)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 28, Finished, Available, Finished)

+------------------+-----+
|tweet_id          |count|
+------------------+-----+
|612278847216574464|2    |
|612279842705268737|2    |
|612332723135385600|2    |
|612335209632002049|2    |
|612394060943220737|2    |
|612394724368908288|2    |
|612395054951305216|2    |
|612405167288225794|2    |
|612458770375905280|2    |
|612502653277356032|2    |
|612285975297830912|2    |
|612361236601135104|2    |
|612380301247930368|2    |
|612501326002438148|2    |
|612342007667625984|2    |
|612362065924132864|2    |
|612416441451544576|2    |
|612235140668878848|2    |
|612280506093170688|2    |
|612304044095225856|2    |
+------------------+-----+
only showing top 20 rows



In [27]:
duplicate_count = (
    df.groupBy("tweet_id")
      .count()
      .filter(col("count") > 1)
      .count()
)

print("Number of tweet_ids with duplicates:", duplicate_count)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 29, Finished, Available, Finished)

Number of tweet_ids with duplicates: 1392


Looking at one duplicate example (`tweet_id` = "612278847216574464") shows:

- Same `external_author_id`

- Same `author`

- Identical tweet `content`

- Same `publish_date` and `harvested date`

- Same `followers` / `following / `updates`

- Same `account_type` and `category`

- Same `article_url`

- Same `region`

In other words, they are **perfect row-level duplicates**.
The tweet appears twice because Twitter released multiple CSV files, and the same record is present in more than one.

Since duplicates are truly identical, we safely remove them.

In [28]:
df.filter(col("tweet_id") == "612278847216574464").show(truncate=False)


StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 30, Finished, Available, Finished)

+------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------+-------------+--------+-------------------+-------------------+---------+---------+-------+------------+-------------+---------------+------------------+--------------------------------------------------------+-------+----------------+-------+
|external_author_id|author   |content                                                                                                                                   |region       |language|publish_date       |harvested_date     |following|followers|updates|account_type|new_june_2018|alt_external_id|tweet_id          |article_url                                             |retweet|account_category|n_links|
+------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------+-----

In [29]:
df = df.dropDuplicates(["tweet_id"])
df.groupBy("tweet_id").count().filter(col("count") > 1).count()

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 31, Finished, Available, Finished)

0

## 11. Checking and cleaning specific columns with inconsistences

The region column requires standardisation because the raw values contain **inconsistent formatting, mixed casing, and several long or outdated country names**. Examples includ entries like "Russian Federation," "Iran, Islamic Republic of," and variants with more spaces. These discrepancies would lower the quality of any subsequent analysis or modeling and produce spurious categories.

### Cleaning `region` column

In [30]:
df.select("region").distinct().orderBy("region").show(200, truncate=False)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 32, Finished, Available, Finished)

+-------------------------+
|region                   |
+-------------------------+
|afghanistan              |
|austria                  |
|azerbaijan               |
|belarus                  |
|canada                   |
|czech republic           |
|denmark                  |
|egypt                    |
|estonia                  |
|finland                  |
|france                   |
|germany                  |
|greece                   |
|hong kong                |
|india                    |
|iran, islamic republic of|
|iraq                     |
|israel                   |
|italy                    |
|japan                    |
|latvia                   |
|malaysia                 |
|mexico                   |
|russian federation       |
|samoa                    |
|saudi arabia             |
|serbia                   |
|spain                    |
|sweden                   |
|switzerland              |
|turkey                   |
|ukraine                  |
|united arab emirate

To resolve this, the values are **first normalised by converting everything to lowercase and removing unnecessary spaces.** In order to ensure that entries like "russian federation" became "russia" and "iran, islamic republic of" became "iran," many nations with lengthy or unusual names are then mapped to their normal modern forms. In order to ensure that all area names follow a clear, model-friendly format, all residual spaces are substituted with underscores (e.g., "united states" → "united_states," "czech republic" → "czech_republic").

In [31]:
# 1. Lowercase and trim (already done earlier)
df = df.withColumn("region", lower(col("region")))
df = df.withColumn("region", regexp_replace(col("region"), " +", " "))

# 2. Standardize specific country names
df = df.withColumn(
    "region",
    when(col("region") == "russian federation", "russia")
    .when(col("region") == "iran, islamic republic of", "iran")
    .when(col("region") == "united states", "united_states")
    .when(col("region") == "united kingdom", "united_kingdom")
    .when(col("region") == "united arab emirates", "uae")
    .otherwise(col("region"))
)

# 3. Replace spaces with underscores to avoid issues in ML later
df = df.withColumn("region", regexp_replace(col("region"), " ", "_"))

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 33, Finished, Available, Finished)

Following these changes, the `region` column now has a consistent and organized collection of categories that may be used for preprocessing and analysis.

In [32]:
df.select("region").distinct().orderBy("region").show(200, truncate=False)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 34, Finished, Available, Finished)

+--------------+
|region        |
+--------------+
|afghanistan   |
|austria       |
|azerbaijan    |
|belarus       |
|canada        |
|czech_republic|
|denmark       |
|egypt         |
|estonia       |
|finland       |
|france        |
|germany       |
|greece        |
|hong_kong     |
|india         |
|iran          |
|iraq          |
|israel        |
|italy         |
|japan         |
|latvia        |
|malaysia      |
|mexico        |
|russia        |
|samoa         |
|saudi_arabia  |
|serbia        |
|spain         |
|sweden        |
|switzerland   |
|turkey        |
|uae           |
|ukraine       |
|united_kingdom|
|united_states |
|unknown       |
+--------------+



### Cleaning `language` column

There are numerous irregularities in the original language field, including multi-word labels, languages written in parenthesis, and values with spaces. The labels are standardized so that the column would be trustworthy for further investigation.

In [33]:
#Inspect the unique language values
df.select("language").distinct().orderBy("language").show(200, truncate=False)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 35, Finished, Available, Finished)

+-------------------+
|language           |
+-------------------+
|albanian           |
|arabic             |
|bengali            |
|bulgarian          |
|catalan            |
|croatian           |
|czech              |
|danish             |
|dutch              |
|english            |
|estonian           |
|farsi (persian)    |
|finnish            |
|french             |
|german             |
|greek              |
|gujarati           |
|hebrew             |
|hindi              |
|hungarian          |
|icelandic          |
|indonesian         |
|italian            |
|japanese           |
|kannada            |
|korean             |
|kurdish            |
|latvian            |
|lithuanian         |
|macedonian         |
|malay              |
|malayalam          |
|norwegian          |
|polish             |
|portuguese         |
|pushto             |
|romanian           |
|russian            |
|serbian            |
|simplified chinese |
|slovak             |
|slovenian          |
|somali   

Some languages, including "tagalog (Filipino)" and "farsi (Persian)," are rebuilt into more **straightforward, distinctive identifiers**. Additionally, multiword languages such as traditional and simplified Chinese are harmonized. Following these modifications, underscores are used to fill in any remaining gaps, making the column completely compatible with downstream processing and machine-learning pipelines.

As a result, there are no redundant or unclear categories, and the collection of linguistic values is clear and consistent.

In [34]:
# Fix specific multi-word or parenthesis languages
df = df.withColumn(
    "language",
    when(col("language") == "farsi (persian)", "farsi")
    .when(col("language") == "tagalog (filipino)", "tagalog")
    .when(col("language") == "simplified chinese", "chinese_simplified")
    .when(col("language") == "traditional chinese", "chinese_traditional")
    .otherwise(col("language"))
)

# Replace spaces with underscores globally
df = df.withColumn("language", regexp_replace(col("language"), " ", "_"))

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 36, Finished, Available, Finished)

In [35]:
df.select("language").distinct().orderBy("language").show(200, truncate=False)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 37, Finished, Available, Finished)

+-------------------+
|language           |
+-------------------+
|albanian           |
|arabic             |
|bengali            |
|bulgarian          |
|catalan            |
|chinese_simplified |
|chinese_traditional|
|croatian           |
|czech              |
|danish             |
|dutch              |
|english            |
|estonian           |
|farsi              |
|finnish            |
|french             |
|german             |
|greek              |
|gujarati           |
|hebrew             |
|hindi              |
|hungarian          |
|icelandic          |
|indonesian         |
|italian            |
|japanese           |
|kannada            |
|korean             |
|kurdish            |
|latvian            |
|lithuanian         |
|macedonian         |
|malay              |
|malayalam          |
|norwegian          |
|polish             |
|portuguese         |
|pushto             |
|romanian           |
|russian            |
|serbian            |
|slovak             |
|slovenian

### Cleaning `account_type` column

The `account_type` field contains inconsistent and noisy values. Many entries correspond to languages or countries rather than actual account types, and others contain typos or null values. The following transformations are applied to standardise the column.

In [36]:
df.select("account_type").distinct().orderBy("account_type").show(200, truncate=False)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 38, Finished, Available, Finished)

+------------+
|account_type|
+------------+
|arabic      |
|commercial  |
|ebola       |
|french      |
|german      |
|hashtager   |
|italian     |
|koch        |
|left        |
|local       |
|news        |
|portuguese  |
|right       |
|russian     |
|spanish     |
|ukranian    |
|unknown     |
|uzbek       |
|zaporoshia  |
+------------+



Several values are actually languages (e.g., arabic, french, german, uzbek, ukrainian, zaporoshia) and not true account types.
These are recoded into a unified category:

In [37]:
df = df.withColumn(
    "account_type",
    when(col("account_type") == "ukranian", "ukrainian")
    .when(col("account_type") == "hashtager", "hashtag")
    .otherwise(col("account_type"))
)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 39, Finished, Available, Finished)

In [38]:
df = df.withColumn(
    "account_type",
    when(col("account_type").isin(
        "arabic", "french", "german", "italian", "portuguese",
        "spanish", "uzbek", "ukrainian", "russian", "zaporoshia"
    ),
         "language_" + col("account_type"))
    .otherwise(col("account_type"))
)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 40, Finished, Available, Finished)

In [39]:
df.select("account_type").distinct().orderBy("account_type").show(200, truncate=False)


StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 41, Finished, Available, Finished)

+------------+
|account_type|
+------------+
|NULL        |
|commercial  |
|ebola       |
|hashtag     |
|koch        |
|left        |
|local       |
|news        |
|right       |
|unknown     |
+------------+



There are 820,977 NULL values in `account_type`.
These are replaced with "unknown" to ensure consistency.

In [40]:
df.filter(col("account_type").isNull()).count()


StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 42, Finished, Available, Finished)

820977

After cleaning, the dataset contains four meaningful account types, "unknown" and additional special cases

In [41]:
df = df.withColumn(
    "account_type",
    when(col("account_type").isNull(), "unknown")
    .otherwise(col("account_type"))
)
df.select("account_type").distinct().orderBy("account_type").show(200, truncate=False)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 43, Finished, Available, Finished)

+------------+
|account_type|
+------------+
|commercial  |
|ebola       |
|hashtag     |
|koch        |
|left        |
|local       |
|news        |
|right       |
|unknown     |
+------------+



### Cleaning `account_category` column

**Typos, concatenated terms, and missing values** are among the inconsistent labels found in the `account_category` field. First, all values are cut and lowercased. Underscores are used to standardize multi-word categories, and specific changes are made (e.g., _lefttroll → left_troll, righttroll → right_troll, fearmonger → fear_monger_). To maintain consistency, "unknown" is used in place of null entries.

In [42]:
df.select("account_category").distinct().orderBy("account_category").show(200, truncate=False)

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 44, Finished, Available, Finished)

+----------------+
|account_category|
+----------------+
|commercial      |
|fearmonger      |
|hashtaggamer    |
|lefttroll       |
|newsfeed        |
|nonenglish      |
|righttroll      |
|unknown         |
+----------------+



### Validating `harvested_date` vs. `publish_date`

We check to see whether any tweets have a harvested_date earlier than the publish_date, which would be impossible and suggest faulty timestamps, in order to guarantee data consistency.

The result of 0 indicates that **all records are in the right chronological order**, since the tweet is always gathered on or after its publish date.

In [43]:
df.filter(col("harvested_date") < col("publish_date")).count()

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 45, Finished, Available, Finished)

0

## 12. Saving the Cleaned Dataset (Parquet & Delta Formats)

Note: Code to Save as Parquet file

In [44]:
#df.repartition(1).write.mode("overwrite").parquet("Files/cleaned_parquet_single")

#print("Single Parquet file created in: Files/cleaned_parquet_single")

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 46, Finished, Available, Finished)

Note: Code to Save as Delta Table

In [45]:
#output_path = "Tables/cleaned_IRA_tweets"

#(
#    df
#   .write
#    .format("delta")
#    .mode("overwrite")
#    .saveAsTable("cleaned_IRA_tweets")
#)

#print("Saved cleaned dataset as a Delta Table.")

StatementMeta(, 5058c8aa-760f-42fc-aed9-7b0e6fa03ab9, 47, Finished, Available, Finished)